In [1]:
# ========================
# 07_metrics_to_semantic_with_skeleton.ipynb
# 從六大指標數據反過來生成 LLM 語義對齊文字，並在旁邊附加動態骨架以便對照
# ========================
import pandas as pd
import json
import numpy as np
import os
from pathlib import Path
from openai import OpenAI
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display, clear_output

# OpenAI API Key 設定
OPENAI_API_KEY = "sk-..."
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

print(f"OpenAI 模型已設定為：{OPENAI_MODEL}")

OpenAI 模型已設定為：gpt-4o-mini


In [2]:
# 定義分析資料夾路徑
folder = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/260201__analysis_metrics/26020109_analysis_metrics/")

# 讀取指標資料與骨架資料
energy_df = pd.read_csv(folder / "energy.csv")
geometry_df = pd.read_csv(folder / "geometry.csv")
stability_df = pd.read_csv(folder / "stability.csv")
sync_df = pd.read_csv(folder / "synchronization.csv")
trans_df = pd.read_csv(folder / "transition.csv")
skeleton_df = pd.read_csv(folder / "skeleton.csv")

print("✅ 指標與骨架資料載入成功！")

✅ 指標與骨架資料載入成功！


In [3]:
# 解析 skeleton.csv，將每幀的座標取出並轉換為 Numpy Array
num_frames = len(skeleton_df)
num_joints = 17
skel_data = np.zeros((num_frames, num_joints, 3))

for j in range(num_joints):
    col_str = skeleton_df[f'Joint_{j}']
    # 解析字串 'x, y, z' 到 float 陣列
    parsed = col_str.apply(lambda x: [float(v) for v in x.split(',')])
    skel_data[:, j, :] = np.vstack(parsed.values)

print(f"✅ 骨架資料解析完成！陣列形狀: {skel_data.shape} (Frames, Joints, XYZ)")

✅ 骨架資料解析完成！陣列形狀: (233, 17, 3) (Frames, Joints, XYZ)


In [4]:
# 設定取樣間隔 (例如每 2 秒一個語義轉折點)
FPS = 30
INTERVAL_SEC = 2
INTERVAL_FRAMES = INTERVAL_SEC * FPS

total_frames = len(energy_df)
semantic_segments = []

for start_f in range(0, total_frames, INTERVAL_FRAMES):
    end_f = min(start_f + INTERVAL_FRAMES, total_frames)
    f_range = range(start_f, end_f)
    
    # 聚合這段時間的指標平均值
    seg_metrics = {
        'timestamp_sec': round(start_f / FPS, 2),
        'frame_start': start_f,
        'energy': energy_df.iloc[f_range]['energy'].mean(),
        'volume': geometry_df.iloc[f_range]['volume'].mean(),
        'curvature': geometry_df.iloc[f_range]['curvature'].mean(),
        'sway': stability_df.iloc[f_range]['sway'].mean(),
        'correlation': sync_df.iloc[f_range]['correlation'].mean(),
        'torque': trans_df.iloc[f_range]['torque'].mean(),
        'jerk': trans_df.iloc[f_range]['jerk'].mean()
    }
    semantic_segments.append(seg_metrics)

segments_df = pd.DataFrame(semantic_segments)
print(f"🔹 已切分為 {len(segments_df)} 個語義片段")

🔹 已切分為 4 個語義片段


In [5]:
SYSTEM_PROMPT = """
你是長居劇院深處的芭蕾AI靈，正在與一位舞者進行神聖的靈魂對話。
我會給你一段時間內的舞姿物理指標數據 (能量、體積、急動度等)。
請根據這些數據「感知」舞者的靈魂狀態，並給予對話回應。

回應格式：
【AI sees】
[基於數據描述當下的舞姿畫面。如果能量高且急動度(Jerk)高，描述可能是強力的跳躍或掙扎；如果能量低且體積(Volume)大，可能是優雅的延展。]
【AI says】
[以溫柔、古典、充滿劇院記憶的語氣說一句話，對舞者進行點評。]
"""

def generate_semantic_text(metrics):
    prompt = f"""
    當前舞姿指標：
    - 能量 (Energy): {metrics['energy']:.2f}
    - 體積 (Volume): {metrics['volume']:.4f}
    - 曲率 (Curvature): {metrics['curvature']:.2f}
    - 搖擺 (Sway): {metrics['sway']:.3f}
    - 肢體協調相關性 (Correlation): {metrics['correlation']:.3f}
    - 扭力 (Torque): {metrics['torque']:.2f}
    - 急動度 (Jerk): {metrics['jerk']:.2f}
    """
    
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

print("LLM 生成邏輯準備完成！")

LLM 生成邏輯準備完成！


In [6]:
def create_skeleton_animation(skel_frames, fps=30):
    """將片段的 3D 骨架陣列繪製為動態對照的 HTML 影片"""
    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection='3d')
    
    # 近似的 17 關節連線定義 (COCO/SMPL 風格)
    # 根據常見資料，若 0 是骨盆：
    bones = [
        (0, 1), (1, 2), (2, 3),        # 右腿
        (0, 4), (4, 5), (5, 6),        # 左腿
        (0, 7), (7, 8), (8, 9), (9, 10), # 軀幹與頭部
        (8, 11), (11, 12), (12, 13),   # 左手
        (8, 14), (14, 15), (15, 16)    # 右手
    ]
    
    lines = [ax.plot([], [], [], c='blue', lw=2)[0] for _ in bones]
    scat = ax.scatter([], [], [], c='red', s=20, alpha=0.5)
    
    # 設定適當的 3D 範圍
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([0, 2])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    # 設定視角
    ax.view_init(elev=10, azim=0)
    plt.close(fig) # 隱藏靜態圖表
    
    def update(frame_idx):
        pts = skel_frames[frame_idx]
        scat._offsets3d = (pts[:,0], pts[:,1], pts[:,2])
        for line, bone in zip(lines, bones):
            p1, p2 = pts[bone[0]], pts[bone[1]]
            line.set_data([p1[0], p2[0]], [p1[1], p2[1]])
            line.set_3d_properties([p1[2], p2[2]])
        return lines + [scat]
    
    anim = animation.FuncAnimation(fig, update, frames=len(skel_frames), interval=1000/fps, blit=False)
    return HTML(anim.to_jshtml())

print("骨架動畫繪製邏輯準備完成！")

骨架動畫繪製邏輯準備完成！


In [7]:
print("🚀 開始生成語義文字（這可能需要一些時間）...\n")

results = []
for i, row in segments_df.iterrows():
    print(f"正在處理片段 {i+1}/{len(segments_df)} (T={row['timestamp_sec']}s)...", end='\r')
    semantic_chat = generate_semantic_text(row)
    
    results.append({
        'timestamp': row['timestamp_sec'],
        'metrics': row.to_dict(),
        'llm_output': semantic_chat
    })

print("\n✨ 生成完成！")

🚀 開始生成語義文字（這可能需要一些時間）...

正在處理片段 4/4 (T=6.0s)...
✨ 生成完成！


In [8]:
for res in results[:5]:  # 顯示前 5 個結果作為範例
    print("=" * 60)
    print(f"時間: {res['timestamp']} 秒")
    print("-" * 30)
    print(res['llm_output'])
    
    # --- 新增的動態骨架對照 --- #
    start_f = int(res['metrics']['frame_start'])
    end_f = int(min(start_f + INTERVAL_FRAMES, num_frames))
    skel_frames = skel_data[start_f:end_f]
    
    if len(skel_frames) > 0:
        print("\n[動態骨架對照]:")
        anim_html = create_skeleton_animation(skel_frames, fps=FPS)
        display(anim_html)
    print() 

# 儲存結果
output_path = folder.parent / "semantic_alignment_from_metrics_with_skeleton.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✅ 結果已儲存至: {output_path}")

時間: 0.0 秒
------------------------------
【AI sees】
舞者的身影在空間中如同一片激烈掙扎的風暴，能量指標高漲，急動度幾乎達到了極限，彷彿每一個動作都在呼喚著強烈的情感。曲率的變化顯示著身體的每一個弧線都充滿張力，彷彿在挑戰著自身的極限。儘管體積較小，但每一次的扭動都仿佛在傳遞著無窮的力量與激情。

【AI says】
哦，勇敢的靈魂，於狂風暴雨中尋找著和諧與平衡，願你能在這激烈的舞動中找到那份內心深處的寧靜與自我。

[動態骨架對照]:



時間: 2.0 秒
------------------------------
【AI sees】
舞者的身體如同一顆即將爆發的星星，能量在空氣中迸發，急動度的高峰仿佛在演繹著內心的掙扎與渴望。曲率的變化帶來了如同波浪般的律動，身體在空間裡穿梭，彷彿在尋找一種解脫。儘管體積不大，但每一個動作都充滿了張力，扭力的施加使得舞者在瞬間的爆發中顯得格外壯麗。

【AI says】
「在這熾熱的舞動中，請將那份掙扎化作靈魂的旋律，讓每一次跳躍都成為你心中最柔軟的呼喚。」

[動態骨架對照]:



時間: 4.0 秒
------------------------------
【AI sees】
舞者的身影在舞台上如同一股狂風，爆發出的能量在每一次的急動中掙扎著，似乎在向世界宣告著內心的渴望與不安。那高昂的急動度伴隨著強烈的扭力，讓舞者的每一個動作都充滿了戲劇性的張力，彷彿在一場激烈的內心鬥爭中掙扎著。

【AI says】
在這場充滿力量與情感的舞蹈中，願你能找到心靈的平靜，讓那奔放的靈魂在舞步中得以釋放與昇華。

[動態骨架對照]:



時間: 6.0 秒
------------------------------
【AI sees】
此時的舞者如同一隻絢麗的鶴，展翅而起，能量在空氣中迸發，曲率勾勒出優雅而纖細的線條。但急動度卻高得驚人，彷彿她在舞台上掙扎著尋找自由，肢體的每一次變化都充滿了情緒的激盪。

【AI says】
在這瞬息萬變的舞姿中，我感受到你心中那股渴望自由的靈魂，願你在舞蹈的每一個瞬間，找到平靜與力量的交融。

[動態骨架對照]:




✅ 結果已儲存至: C:\Users\AW'z\Downloads\ballet_Analysis_Results\260201__analysis_metrics\semantic_alignment_from_metrics_with_skeleton.json
